In [6]:
import numpy as np
import polars as pl
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

n = 1000
df = pl.DataFrame({
    "f1": np.random.rand(n),
    "f2": np.random.rand(n),
    "f3": np.random.rand(n),
    "cat": np.random.choice(["A", "B", "C"], n),
    "target": np.random.rand(n)
})

df = df.with_columns((pl.col("f1") * pl.col("f2")).alias("interaction"))

X = df.drop("target").to_pandas()
y = df["target"].to_numpy()

num_features = ["f1", "f2", "f3", "interaction"]
cat_features = ["cat"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(objective='reg:squarederror'))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    "regressor__n_estimators": [50, 100],
    "regressor__max_depth": [3, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=3)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
preds = best_model.predict(X_test)

joblib.dump(best_model, "pipeline.pkl")

print(f"R2 Score: {r2_score(y_test, preds):.2f}")
print(f"MSE: {mean_squared_error(y_test, preds):.2f}")
print(df.head())

R2 Score: -0.08
MSE: 0.09
shape: (5, 6)
┌──────────┬──────────┬──────────┬─────┬──────────┬─────────────┐
│ f1       ┆ f2       ┆ f3       ┆ cat ┆ target   ┆ interaction │
│ ---      ┆ ---      ┆ ---      ┆ --- ┆ ---      ┆ ---         │
│ f64      ┆ f64      ┆ f64      ┆ str ┆ f64      ┆ f64         │
╞══════════╪══════════╪══════════╪═════╪══════════╪═════════════╡
│ 0.719079 ┆ 0.098575 ┆ 0.668051 ┆ B   ┆ 0.271844 ┆ 0.070884    │
│ 0.88273  ┆ 0.202669 ┆ 0.083506 ┆ B   ┆ 0.076174 ┆ 0.178902    │
│ 0.21259  ┆ 0.019782 ┆ 0.355676 ┆ C   ┆ 0.720314 ┆ 0.004206    │
│ 0.78147  ┆ 0.560227 ┆ 0.78148  ┆ A   ┆ 0.3088   ┆ 0.4378      │
│ 0.689179 ┆ 0.376981 ┆ 0.941537 ┆ B   ┆ 0.532936 ┆ 0.259807    │
└──────────┴──────────┴──────────┴─────┴──────────┴─────────────┘
